Identical split to training: seed 42, 90/10

In [1]:
import json, random, math, os, gc
import numpy as np, pandas as pd, torch
 
DATASET_PATH = "../data/tanglish.jsonl"
ADAPTER_DIR  = "../tanglish_gemma2b_adapter"
BASE_ID      = "google/gemma-2b"
IT_ID        = "google/gemma-2b-it"
MAX_NEW      = 200
RESULTS_DIR  = "../results"
os.makedirs(RESULTS_DIR, exist_ok=True)
 
with open(DATASET_PATH, "r", encoding="utf-8") as f:
    raw = [json.loads(l) for l in f]
 
def fmt(r):
    ins, inp, out = (r.get("instruction", "").strip(), r.get("input", "").strip(), r.get("output", "").strip())
    user = f"{ins}\n\nContext:\n{inp}" if inp else ins
    return {
        "text": f"<start_of_turn>user\n{user}<end_of_turn>\n<start_of_turn>model\n{out}<end_of_turn>",
        "instruction": ins, "input": inp, "output": out,
    }
 
formatted = [fmt(r) for r in raw]
random.seed(42)
random.shuffle(formatted)
cut = int(len(formatted) * 0.9)
train, val = formatted[:cut], formatted[cut:]
print(f"Train {len(train)} | Val {len(val)}")
 
MODEL_TAG = "<start_of_turn>model\n"
def split_prompt_target(text):
    prompt = text.split(MODEL_TAG)[0] + MODEL_TAG
    target = text.split(MODEL_TAG)[-1].replace("<end_of_turn>", "").strip()
    return prompt, target
 
val_prompts, val_refs = zip(*[split_prompt_target(v["text"]) for v in val])
val_prompts, val_refs = list(val_prompts), list(val_refs)


Train 551 | Val 62


Fuzzy leakage check (near-duplicates, not just exact)

In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
 
def near_dup_report(train_texts, val_texts, name, thr=0.90):
    vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5)).fit(train_texts + val_texts)
    sims = cosine_similarity(vec.transform(val_texts), vec.transform(train_texts))
    best = sims.max(axis=1)
    print(f"[{name}] mean best-sim {best.mean():.3f} | max {best.max():.3f} | "
          f"val items with sim >= {thr}: {(best >= thr).sum()}/{len(best)}")
    return best
 
best_prompt = near_dup_report([t["instruction"] + " " + t["input"] for t in train],
                              [v["instruction"] + " " + v["input"] for v in val], "prompts")
best_output = near_dup_report([t["output"] for t in train],
                              [v["output"] for v in val], "outputs")
pd.DataFrame({"prompt_sim": best_prompt, "output_sim": best_output}).to_csv(
    f"{RESULTS_DIR}/leakage_fuzzy.csv", index=False)


[prompts] mean best-sim 0.319 | max 0.561 | val items with sim >= 0.9: 0/62
[outputs] mean best-sim 0.305 | max 0.625 | val items with sim >= 0.9: 0/62


Load models: base+LoRA (one object, adapter toggled) and Gemma-2B-it

In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
 
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)
tok = AutoTokenizer.from_pretrained(BASE_ID)
tok.pad_token = tok.eos_token
END_ID = tok.convert_tokens_to_ids("<end_of_turn>")
 
base = AutoModelForCausalLM.from_pretrained(BASE_ID, quantization_config=bnb, device_map="auto")
ft_model = PeftModel.from_pretrained(base, ADAPTER_DIR)
ft_model.eval()
 
it_model = AutoModelForCausalLM.from_pretrained(IT_ID, quantization_config=bnb, device_map="auto")
it_model.eval()
 


/home/prathoseraajv/cross_lingual/.venv/lib64/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] We found `hidden_act="gelu"` in this Gemma config. This is a legacy value of the official releases but it is meant to target the tanh approximation. Setting `hidden_act="gelu_pytorch_tanh"` instead.
Loading weights: 100%|██████████| 164/164 [00:03<00:00, 54.13it/s]


GemmaForCausalLM(
  (model): GemmaModel(
    (embed_tokens): GemmaTextScaledWordEmbedding(256000, 2048, padding_idx=0)
    (layers): ModuleList(
      (0-17): 18 x GemmaDecoderLayer(
        (self_attn): GemmaAttention(
          (q_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear4bit(in_features=2048, out_features=256, bias=False)
          (v_proj): Linear4bit(in_features=2048, out_features=256, bias=False)
          (o_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): GemmaMLP(
          (gate_proj): Linear4bit(in_features=2048, out_features=16384, bias=False)
          (up_proj): Linear4bit(in_features=2048, out_features=16384, bias=False)
          (down_proj): Linear4bit(in_features=16384, out_features=2048, bias=False)
          (act_fn): GELUTanh()
        )
        (input_layernorm): GemmaRMSNorm((2048,), eps=1e-06)
        (post_attention_layernorm): GemmaRMSNorm((2048,), eps=1e-06)
      )

Core functions: response-only NLL (token-weighted) and generation

In [4]:
import torch.nn.functional as F
 
@torch.no_grad()
def response_nll(model, text, prompt):
    """Return (sum NLL over response tokens, number of response tokens)."""
    ids = tok(text, return_tensors="pt").input_ids.to("cuda")
    p_len = len(tok(prompt).input_ids)
    logits = model(input_ids=ids).logits[:, :-1, :].float()
    targets = ids[:, 1:]
    nll = F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1), reduction="none")
    nll = nll.reshape(targets.shape)[:, p_len - 1:]        # response tokens only
    return nll.sum().item(), nll.numel()
 
@torch.no_grad()
def generate(model, prompt):
    inp = tok(prompt, return_tensors="pt").to("cuda")
    out = model.generate(**inp, max_new_tokens=MAX_NEW, do_sample=False,
                         eos_token_id=[tok.eos_token_id, END_ID], pad_token_id=tok.pad_token_id)
    return tok.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True).strip()
 
def run_model(name, model, use_adapter=None):
    """use_adapter: None (plain model), True (adapter on), False (adapter off)."""
    nll_sum, ntok, preds = [], [], []
    for v, prompt in zip(val, val_prompts):
        if use_adapter is False:
            with model.disable_adapter():
                s, n = response_nll(model, v["text"], prompt)
                p = generate(model, prompt)
        else:
            s, n = response_nll(model, v["text"], prompt)
            p = generate(model, prompt)
        nll_sum.append(s); ntok.append(n); preds.append(p)
    print(f"{name}: done")
    return {"nll_sum": np.array(nll_sum), "ntok": np.array(ntok), "preds": preds}

Run all three systems on the validation set

In [5]:
runs = {
    "base":        run_model("base", ft_model, use_adapter=False),
    "finetuned":   run_model("finetuned", ft_model, use_adapter=True),
    "gemma2b_it":  run_model("gemma2b_it", it_model, use_adapter=None),
}

base: done
finetuned: done
gemma2b_it: done


In [6]:
import re
 
affected = [i for i, p in enumerate(runs["finetuned"]["preds"]) if "<unused" in p]
print("Fine-tuned outputs containing <unusedN>:", affected)
 
# What does the model actually emit at the end? Look at the raw last tokens.
@torch.no_grad()
def raw_tail(model, prompt, k=6):
    inp = tok(prompt, return_tensors="pt").to("cuda")
    out = model.generate(**inp, max_new_tokens=MAX_NEW, do_sample=False,
                         eos_token_id=[tok.eos_token_id, END_ID], pad_token_id=tok.pad_token_id)
    ids = out[0][inp["input_ids"].shape[1]:].tolist()
    return tok.convert_ids_to_tokens(ids[-k:]), len(ids)
 
for i in affected[:4]:
    print(i, raw_tail(ft_model, val_prompts[i]))
 
# How do training examples end after tokenization? (compare with the above)
print("Train example tail tokens:", tok.convert_ids_to_tokens(tok(train[0]["text"]).input_ids[-5:]))
 
def clean(t):
    t = re.sub(r"<unused\d+>", "", t)
    t = t.replace("<end_of_turn>", "").replace("<eos>", "")
    return t.strip()
 
for r in runs.values():                 # idempotent, safe to re-run
    r["preds"] = [clean(p) for p in r["preds"]]
 
# %% [N2] New systems: prompted Gemma-2B-it (fairer baseline) + no-repeat decoding
INSTR = ("Answer in Tanglish: romanized Tamil (Tamil written in English letters) mixed with English "
         "technical terms, the way a Tamil developer would explain it to a friend. "
         "Put code and technical terms in backticks. Do not answer in English sentences.")
 
rs = random.Random(7)
cands = [t for t in train if len(t["text"]) < 900]
shots = rs.sample(cands, 3)            # few-shot examples come from TRAIN only (no val leakage)
 
def instr_prompt(p):                   # zero-shot with instruction
    return p.replace("<start_of_turn>user\n", "<start_of_turn>user\n" + INSTR + "\n\n", 1)
 
def fewshot_prompt(p):                 # instruction + 3 train examples
    ex = ""
    for i, s in enumerate(shots):
        t = s["text"]
        if i == 0:
            t = t.replace("<start_of_turn>user\n", "<start_of_turn>user\n" + INSTR + "\n\n", 1)
        ex += t + "\n"
    return ex + p
 
@torch.no_grad()
def generate_kw(model, prompt, **kw):
    inp = tok(prompt, return_tensors="pt").to("cuda")
    out = model.generate(**inp, max_new_tokens=MAX_NEW, do_sample=False,
                         eos_token_id=[tok.eos_token_id, END_ID], pad_token_id=tok.pad_token_id, **kw)
    return clean(tok.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True))
 
def run_prompted(name, model, builder):
    nll_sum, ntok, preds = [], [], []
    for v, prompt in zip(val, val_prompts):
        full, pr = builder(v["text"]), builder(prompt)
        s, n = response_nll(model, full, pr)
        nll_sum.append(s); ntok.append(n); preds.append(generate_kw(model, pr))
    print(name, "done")
    return {"nll_sum": np.array(nll_sum), "ntok": np.array(ntok), "preds": preds}
 
runs["gemma2b_it_instr"]     = run_prompted("gemma2b_it_instr", it_model, instr_prompt)
runs["gemma2b_it_fewshot3"]  = run_prompted("gemma2b_it_fewshot3", it_model, fewshot_prompt)
 
# Fine-tuned with repetition control (PPL is unchanged, only generation differs)
runs["finetuned_norepeat"] = {
    "nll_sum": runs["finetuned"]["nll_sum"], "ntok": runs["finetuned"]["ntok"],
    "preds": [generate_kw(ft_model, p, no_repeat_ngram_size=4) for p in val_prompts],
}
print("Now RE-RUN your existing Cell 6 and Cell 7.")


Fine-tuned outputs containing <unusedN>: [1, 12, 17, 18, 24, 25, 30, 34, 35, 40, 54, 57, 58]
1 (['▁a', 'ag', 'um', '.', '<unused3>', '<eos>'], 192)
12 (['ru', 'kk', 'um', '.', '<unused3>', '<eos>'], 159)
17 (['▁use', '▁pann', 'anum', '.', '<unused3>', '<eos>'], 186)
18 (['-', 'ku', '▁perfect', '.', '<unused3>', '<eos>'], 193)
Train example tail tokens: ['ru', 'kk', 'um', '.', '<end_of_turn>']


[W1023 15:11:44.574878423 CUDACachingAllocator.cpp:3934] memory allocation failed with OOM on device 0 while trying to allocate 322961408 bytes (free: 314966016, total: 8177909760).


gemma2b_it_instr done


[W1023 15:13:06.508585076 CUDACachingAllocator.cpp:3934] memory allocation failed with OOM on device 0 while trying to allocate 884998144 bytes (free: 48627712, total: 8177909760).
[W1023 15:13:12.610363420 CUDACachingAllocator.cpp:3934] memory allocation failed with OOM on device 0 while trying to allocate 918552576 bytes (free: 88473600, total: 8177909760).
[W1023 15:14:08.376579967 CUDACachingAllocator.cpp:3934] memory allocation failed with OOM on device 0 while trying to allocate 939524096 bytes (free: 4587520, total: 8177909760).
[W1023 15:14:31.197360096 CUDACachingAllocator.cpp:3934] memory allocation failed with OOM on device 0 while trying to allocate 962592768 bytes (free: 822476800, total: 8177909760).
[W1023 15:14:56.771882724 CUDACachingAllocator.cpp:3934] memory allocation failed with OOM on device 0 while trying to allocate 981467136 bytes (free: 757465088, total: 8177909760).


gemma2b_it_fewshot3 done
Now RE-RUN your existing Cell 6 and Cell 7.


Per-sample metrics: ROUGE, chrF, BERTScore

In [7]:
import evaluate
from sacrebleu.metrics import CHRF
 
rouge = evaluate.load("rouge")
chrf = CHRF()
 
def per_sample_metrics(preds, refs):
    r = rouge.compute(predictions=preds, references=refs, use_aggregator=False)
    out = {"rouge1": np.array(r["rouge1"]), "rouge2": np.array(r["rouge2"]), "rougeL": np.array(r["rougeL"]),
           "chrf": np.array([chrf.sentence_score(p, [t]).score for p, t in zip(preds, refs)])}
    try:
        from bert_score import score as bscore
        _, _, f1 = bscore(preds, refs, model_type="bert-base-multilingual-cased", verbose=False)
        out["bertscore_f1"] = f1.numpy()
    except Exception as e:
        print("BERTScore skipped:", e)
    return out
 
for k, r in runs.items():
    r["metrics"] = per_sample_metrics(r["preds"], val_refs)


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1111.31it/s]
[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 23390.50it/s]
[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                   

Bootstrap CIs + paired significance

In [8]:
rng = np.random.default_rng(0)
B = 10000
N = len(val)
idx_boot = rng.integers(0, N, size=(B, N))   # same resamples for every system -> paired
 
def ppl_from(sums, toks, idx):
    return np.exp(sums[idx].sum(axis=1) / toks[idx].sum(axis=1))
 
def ci(arr):
    return float(np.percentile(arr, 2.5)), float(np.percentile(arr, 97.5))
 
rows = []
for name, r in runs.items():
    full = math.exp(r["nll_sum"].sum() / r["ntok"].sum())
    lo, hi = ci(ppl_from(r["nll_sum"], r["ntok"], idx_boot))
    rows.append({"system": name, "metric": "response_PPL (lower=better)", "value": full, "ci_low": lo, "ci_high": hi})
    for m, vals in r["metrics"].items():
        lo, hi = ci(vals[idx_boot].mean(axis=1))
        rows.append({"system": name, "metric": m, "value": float(vals.mean()), "ci_low": lo, "ci_high": hi})
 
summary = pd.DataFrame(rows)
summary.to_csv(f"{RESULTS_DIR}/rq1_summary_with_ci.csv", index=False)
print(summary.round(4).to_string(index=False))
 
# Paired bootstrap: finetuned vs each baseline
def paired(a, b, label):
    out = []
    pa = ppl_from(runs[a]["nll_sum"], runs[a]["ntok"], idx_boot)
    pb = ppl_from(runs[b]["nll_sum"], runs[b]["ntok"], idx_boot)
    d = pb - pa    # PPL: positive = a (finetuned) better
    out.append({"comparison": label, "metric": "PPL (b - a)", "mean_diff": d.mean(), "ci_low": ci(d)[0],
                "ci_high": ci(d)[1], "p_a_not_better": float((d <= 0).mean())})
    for m in runs[a]["metrics"]:
        d = (runs[a]["metrics"][m][idx_boot].mean(axis=1) - runs[b]["metrics"][m][idx_boot].mean(axis=1))
        out.append({"comparison": label, "metric": m + " (a - b)", "mean_diff": d.mean(), "ci_low": ci(d)[0],
                    "ci_high": ci(d)[1], "p_a_not_better": float((d <= 0).mean())})
    return out
 
paired_df = pd.DataFrame(paired("finetuned", "base", "finetuned vs base") +
                         paired("finetuned", "gemma2b_it", "finetuned vs gemma2b_it"))
paired_df.to_csv(f"{RESULTS_DIR}/rq1_paired_tests.csv", index=False)
print("\n", paired_df.round(4).to_string(index=False))
# Reading it: if the CI of the difference excludes 0, the gain is statistically supported.
 
# Optional: re-run on low-similarity subset to show leakage does not drive results
mask = (best_prompt < 0.90) & (best_output < 0.90)
print(f"\nVal items kept after near-duplicate filter: {mask.sum()}/{N}")
for k, r in runs.items():
    print(k, "PPL (filtered):", round(math.exp(r["nll_sum"][mask].sum() / r["ntok"][mask].sum()), 2),
          "| ROUGE-L (filtered):", round(r["metrics"]["rougeL"][mask].mean(), 4))


             system                      metric    value   ci_low  ci_high
               base response_PPL (lower=better)  71.8689  61.4728  82.6336
               base                      rouge1   0.0840   0.0680   0.1012
               base                      rouge2   0.0117   0.0077   0.0166
               base                      rougeL   0.0693   0.0567   0.0826
               base                        chrf  14.8210  13.1921  16.4834
               base                bertscore_f1   0.5160   0.5032   0.5290
          finetuned response_PPL (lower=better)   8.6172   7.8615   9.4208
          finetuned                      rouge1   0.3246   0.3042   0.3451
          finetuned                      rouge2   0.0772   0.0658   0.0895
          finetuned                      rougeL   0.2005   0.1871   0.2142
          finetuned                        chrf  35.0669  33.1881  36.8611
          finetuned                bertscore_f1   0.7046   0.6937   0.7142
         gemma2b_it respo

In [ ]:
MARKERS = set("""pannum panni pannalaam pannanum pannuvom pannuradhu panna aagum aagura irukku irukkum irukkura
illa illana vechi vachi dhaan oru ovvoru adhu idhu mattum aana appuram mudhalla romba ungal varum pogum
thevai kedaikkum maadhiri adhunaala kammi nalla theriyum kandupidikkum use-pannalaam""".split())
SUFFIX = re.compile(r"[\w`)]-(la|ku|kum|oda|a|um|na|ai)\b")
 
def tanglish_rate(text):
    words = re.findall(r"[a-z]+", text.lower())
    if not words:
        return 0.0
    hits = sum(w in MARKERS for w in words) + len(SUFFIX.findall(text.lower()))
    return hits / len(words)
 
ref_rates = np.array([tanglish_rate(r) for r in val_refs])
THR = float(np.percentile(ref_rates, 5))       # threshold calibrated on the human/reference text
print(f"Reference Tanglish-rate: mean {ref_rates.mean():.3f}, 5th pct (threshold) {THR:.3f}")
 
def rep4(text):
    w = text.split()
    g = [tuple(w[i:i + 4]) for i in range(len(w) - 3)]
    return 0.0 if not g else 1 - len(set(g)) / len(g)
 
rows = []
systems = {"reference": val_refs, **{k: r["preds"] for k, r in runs.items()}}
for name, preds in systems.items():
    rates = np.array([tanglish_rate(p) for p in preds])
    reps = np.array([rep4(p) for p in preds])
    adh = (rates >= THR).astype(float)
    lo, hi = ci(adh[idx_boot].mean(axis=1))
    rows.append({"system": name, "tanglish_adherence": adh.mean(), "adh_ci_low": lo, "adh_ci_high": hi,
                 "mean_repeat4": reps.mean(), "share_degenerate(rep4>0.3)": (reps > 0.3).mean()})
ad_df = pd.DataFrame(rows)
ad_df.to_csv(f"{RESULTS_DIR}/rq1_adherence_repetition.csv", index=False)
print(ad_df.round(3).to_string(index=False))
# Sanity: 'reference' should be ~95%+ adherent and plain English outputs ~0%. If not, tune MARKERS.
 
# Extra paired comparisons against the fairer baselines
extra = pd.DataFrame(
    paired("finetuned", "gemma2b_it_instr", "finetuned vs it+instruction") +
    paired("finetuned", "gemma2b_it_fewshot3", "finetuned vs it+3-shot") +
    paired("finetuned_norepeat", "finetuned", "no-repeat vs greedy"))
extra.to_csv(f"{RESULTS_DIR}/rq1_paired_tests_extended.csv", index=False)
print("\n", extra.round(4).to_string(index=False))
 
# %% [N4] Blind correctness rubric (this is the metric that measures hallucination)
# Rubric: 1 = wrong or degenerate | 2 = partly right (core idea ok, notable errors) | 3 = correct & useful
sample_idx = pd.DataFrame(index=range(len(val_refs))).sample(20, random_state=1).index.tolist()
items = []
for i in sample_idx:
    for s in ["finetuned", "finetuned_norepeat", "gemma2b_it_fewshot3"]:
        items.append({"val_idx": i, "system": s, "prompt": val_prompts[i][-500:],
                      "reference": val_refs[i], "output": runs[s]["preds"][i]})
random.Random(3).shuffle(items)                       # blind: random order, no system labels
sheet = pd.DataFrame(items)
key = sheet[["val_idx", "system"]].copy()
key.insert(0, "item_id", range(len(key)))
sheet = sheet.drop(columns="system")
sheet.insert(0, "item_id", range(len(sheet)))
sheet["correctness_1to3"] = ""
sheet["tanglish_ok_0or1"] = ""
sheet["notes"] = ""
sheet.to_csv(f"{RESULTS_DIR}/rubric_sheet_BLIND.csv", index=False)
key.to_csv(f"{RESULTS_DIR}/rubric_key_DONT_OPEN.csv", index=False)
print("Fill results/rubric_sheet_BLIND.csv (do NOT open the key file). "
      "Make a copy for a second rater if possible.")
 
# After scoring, run this:
def summarize_rubric(sheet_path, key_path=f"{RESULTS_DIR}/rubric_key_DONT_OPEN.csv"):
    s = pd.read_csv(sheet_path).merge(pd.read_csv(key_path), on="item_id")
    s["correctness_1to3"] = pd.to_numeric(s["correctness_1to3"])
    print(s.groupby("system")["correctness_1to3"].agg(["mean", "std", "count"]).round(3))
    print(pd.crosstab(s["system"], s["correctness_1to3"]))
    return s


Reference Tanglish-rate: mean 0.261, 5th pct (threshold) 0.192
             system  tanglish_adherence  adh_ci_low  adh_ci_high  mean_repeat4  share_degenerate(rep4>0.3)
          reference               0.935       0.871        0.984         0.000                       0.000
               base               0.000       0.000        0.000         0.682                       0.855
          finetuned               0.887       0.806        0.952         0.096                       0.113
         gemma2b_it               0.000       0.000        0.000         0.048                       0.000
   gemma2b_it_instr               0.000       0.000        0.000         0.064                       0.032
gemma2b_it_fewshot3               0.306       0.194        0.419         0.050                       0.048
 finetuned_norepeat               0.968       0.919        1.000         0.000                       0.000

                  comparison               metric  mean_diff   ci_low  ci_high  

Save predictions for manual reading (20 random samples)

In [10]:
pred_df = pd.DataFrame({"prompt": val_prompts, "reference": val_refs,
                        **{k: r["preds"] for k, r in runs.items()}})
pred_df.to_csv(f"{RESULTS_DIR}/val_predictions_all_systems.csv", index=False)
pred_df.sample(20, random_state=1).to_csv(f"{RESULTS_DIR}/manual_review_sample.csv", index=False)
print("Saved. Open results/manual_review_sample.csv and score each output 1-5 for "
      "(a) correctness, (b) fluency, (c) Tanglish style.")


Saved. Open results/manual_review_sample.csv and score each output 1-5 for (a) correctness, (b) fluency, (c) Tanglish style.


Save raw per-sample arrays (needed later for Steps 4-5)

In [11]:
import pickle
with open(f"{RESULTS_DIR}/rq1_runs.pkl", "wb") as f:
    pickle.dump({k: {kk: vv for kk, vv in r.items()} for k, r in runs.items()}, f)
